# Chapter 7: Multi-Agent Collaboration

EliteAI Academy

Build specialised agents and compare handoffs, supervision, parallel teams, debate, review, and hierarchical collaboration.

## 0. Setup

Run `setup.ps1` from the course folder once, then select **Python (Agentic AI Course)** as the kernel.
The course loads your DeepSeek credentials from `.env`; restart the kernel after editing that file.
The install cell below is optional if setup is already complete. Run the remaining cells in order.

**Teaching check:** explain the pattern, run one example, inspect its output, then try an exercise.
Live model calls use API credits and may produce different answers. Stock prices, weather, CRM records,
email delivery, and approval workflows in these lessons are classroom simulations. Model-generated
research is illustrative and needs independent source verification before factual use.


In [ ]:
%pip install -r requirements-lock.txt --quiet

In [ ]:
from course_config import MODEL
import os, json, re, time, asyncio
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any

PROVIDER = "deepseek"

if PROVIDER == "deepseek":
    import anthropic
    _client = anthropic.Anthropic(timeout=60.0, max_retries=2)
    _model  = MODEL

    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.",
                 max_tokens: int = 1024) -> str:
        r = _client.messages.create(thinking={"type": "disabled"}, 
            model=_model, max_tokens=max_tokens, system=system,
            messages=[{"role": "user", "content": prompt}]
        )
        return r.content[0].text.strip()

elif PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    _gmodel = genai.GenerativeModel("gemini-2.0-flash")

    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.",
                 max_tokens: int = 1024) -> str:
        return _gmodel.generate_content(f"{system}\n\n{prompt}").text.strip()

async def acall_llm(prompt: str, system: str = "You are a helpful AI assistant.") -> str:
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(None, lambda: call_llm(prompt, system))

print(f"✅ Provider: {PROVIDER}")
print(call_llm("Reply with exactly: MULTI-AGENT READY"))

## 1. The Problem — Monolithic Agents Break Under Complexity

A single agent asked to research, analyse, write, and edit simultaneously degrades in quality. Multi-agent systems divide the labour among specialists.

In [ ]:
COMPLEX_TASK = """
Research the latest trends in renewable energy, analyse the data for investment opportunities,
write a 300-word professional report, review it for factual accuracy, and suggest a 
social media headline to promote it.
"""

print("=== MONOLITHIC AGENT (single prompt) ===")
monolithic_result = call_llm(
    COMPLEX_TASK,
    system="You are a general-purpose AI assistant."
)
print(monolithic_result[:500])
print("\n⚠️  Mixed quality. Hard to verify each domain. No specialisation.")

## 2. Agent Abstraction — The Building Block

We build a reusable `Agent` class that every collaboration pattern will use.

In [ ]:
@dataclass
class Agent:
    'A specialised agent with a defined role, goal, and backstory.'
    name:        str
    role:        str
    goal:        str
    backstory:   str
    tools:       List[str] = field(default_factory=list)
    history:     List[Dict] = field(default_factory=list)
    verbose:     bool = True

    @property
    def system_prompt(self) -> str:
        tools_text = f"\nYou have access to these tools: {', '.join(self.tools)}" if self.tools else ""
        return (
            f"You are {self.name}, a {self.role}.\n"
            f"Your goal: {self.goal}\n"
            f"Background: {self.backstory}"
            f"{tools_text}\n\n"
            f"Always stay in your specialist role. Be precise and professional."
        )

    def run(self, task: str, context: str = "") -> str:
        'Execute a task, optionally with context from previous agents.'
        full_prompt = task
        if context:
            full_prompt = f"Context from previous agents:\n{context}\n\nYour task:\n{task}"

        if self.verbose:
            print(f"\n[{self.name}] 🔄 Working on: {task[:80]}...")

        result = call_llm(full_prompt, system=self.system_prompt)
        self.history.append({"task": task, "result": result})

        if self.verbose:
            print(f"[{self.name}] ✅ Done ({len(result)} chars)")
        return result

    async def arun(self, task: str, context: str = "") -> str:
        'Async version for parallel execution.'
        full_prompt = task if not context else f"Context:\n{context}\n\nTask:\n{task}"
        result = await acall_llm(full_prompt, system=self.system_prompt)
        self.history.append({"task": task, "result": result})
        if self.verbose:
            print(f"[{self.name}] ✅ Async complete")
        return result

    def __repr__(self):
        return f"Agent({self.name}, role={self.role})"

test_agent = Agent(
    name="TestAgent", role="Python tutor",
    goal="Explain Python concepts clearly",
    backstory="10 years teaching programming."
)
print(test_agent.run("What is a list comprehension in Python? One sentence."))

## 3. Collaboration Model 1 — Sequential Handoff (Pipeline)

Output of Agent A → input of Agent B → input of Agent C Each agent builds on the previous one's work.

```
RESEARCHER → ANALYST → WRITER → EDITOR
```

In [ ]:
researcher = Agent(
    name="Researcher",
    role="Senior Research Analyst",
    goal="Find and summarise the latest trends on a topic with specific facts and data.",
    backstory="Expert at synthesising information from multiple sources into clear summaries."
)

analyst = Agent(
    name="Analyst",
    role="Data and Investment Analyst",
    goal="Identify investment opportunities and quantify risks from research findings.",
    backstory="CFA-certified analyst specialising in emerging technology sectors."
)

writer = Agent(
    name="Writer",
    role="Technical Content Writer",
    goal="Write a clear, professional report for a business audience.",
    backstory="Award-winning writer who makes complex topics accessible."
)

editor = Agent(
    name="Editor",
    role="Senior Editor",
    goal="Ensure factual accuracy, clarity, and professional quality.",
    backstory="20 years at top-tier financial publications. Zero tolerance for errors."
)

def sequential_pipeline(topic: str) -> dict:
    'Run a 4-agent sequential pipeline on a topic.'
    print(f"\n{'='*60}")
    print(f"SEQUENTIAL PIPELINE: {topic}")
    print(f"{'='*60}")
    t0 = time.perf_counter()

    research  = researcher.run(f"Research the top 3 trends in: {topic}. Include specific statistics.")

    analysis  = analyst.run(
        f"Identify investment opportunities in: {topic}",
        context=research
    )

    draft = writer.run(
        f"Write a 250-word professional report on: {topic}",
        context=f"Research:\n{research}\n\nAnalysis:\n{analysis}"
    )

    final = editor.run(
        "Review this report for accuracy, clarity, and professional quality. Return the polished version.",
        context=draft
    )

    elapsed = time.perf_counter() - t0
    print(f"\n⏱️  Pipeline completed in {elapsed:.2f}s")

    return {"research": research, "analysis": analysis, "draft": draft, "final": final}

pipeline_result = sequential_pipeline("Renewable Energy and Green Technology")

print(f"\n{'='*60}")
print("FINAL POLISHED REPORT")
print(f"{'='*60}")
print(pipeline_result["final"])

## 4. Collaboration Model 2 — Supervisor-Worker Pattern

A supervisor delegates subtasks to workers, then combines their results.

In [ ]:
class SupervisorAgent:
    'Orchestrator that decomposes a task, delegates to workers,'
    def __init__(self, name: str, workers: List[Agent]):
        self.name    = name
        self.workers = {w.name: w for w in workers}

    def decompose(self, goal: str) -> list:
        'Use LLM to break goal into worker assignments.'
        worker_list = "\n".join(
            f"- {w.name}: {w.role} — {w.goal}" for w in self.workers.values()
        )
        prompt = f"""
You are a project manager. Decompose this goal into specific tasks for each worker.

Goal: {goal}

Available workers:
{worker_list}

Return ONLY a JSON array. Each item must have:
- "worker": exact worker name from the list above
- "task": specific task description for that worker
"""
        raw = call_llm(prompt, system="You are a precise task allocator. Return only valid JSON.")
        return json.loads(re.sub(r"```[\w]*", "", raw).strip())

    def synthesise(self, goal: str, results: dict) -> str:
        'Combine all worker outputs into a final answer.'
        results_text = "\n\n".join(f"[{k}]:\n{v}" for k, v in results.items())
        return call_llm(
            f"Goal: {goal}\n\nWorker outputs:\n{results_text}\n\nSynthesize into a final comprehensive response.",
            system="You are a senior project manager synthesising expert team outputs."
        )

    def run(self, goal: str) -> dict:
        print(f"\n{'='*60}")
        print(f"SUPERVISOR [{self.name}]: {goal}")
        print(f"{'='*60}")

        assignments = self.decompose(goal)
        print(f"\n📋 Task assignments:")
        for a in assignments:
            print(f"   → {a['worker']}: {a['task'][:70]}")

        results = {}
        for assignment in assignments:
            worker_name = assignment["worker"]
            task        = assignment["task"]
            if worker_name in self.workers:
                result = self.workers[worker_name].run(task)
                results[worker_name] = result
            else:
                print(f"   ⚠️  Worker '{worker_name}' not found — skipped")

        print("\n🔗 Supervisor synthesising all outputs...")
        final = self.synthesise(goal, results)

        return {"assignments": assignments, "worker_results": results, "final_output": final}

market_researcher = Agent(
    name="MarketResearcher", role="Market Research Specialist",
    goal="Find market size, growth rate, and key players.",
    backstory="Expert in market intelligence with 15 years of experience."
)
tech_analyst = Agent(
    name="TechAnalyst", role="Technology Analyst",
    goal="Evaluate technical maturity and adoption barriers.",
    backstory="Ex-CTO turned analyst covering emerging technology stacks."
)
risk_assessor = Agent(
    name="RiskAssessor", role="Risk Assessment Specialist",
    goal="Identify and quantify risks and mitigation strategies.",
    backstory="Former hedge fund risk manager with expertise in technology sectors."
)

supervisor = SupervisorAgent(
    name="ChiefAnalyst",
    workers=[market_researcher, tech_analyst, risk_assessor]
)

sv_result = supervisor.run("Produce a comprehensive analysis of the AI chip market in 2025.")

print(f"\n{'='*60}\nFINAL SYNTHESISED REPORT\n{'='*60}")
print(sv_result["final_output"])

## 5. Collaboration Model 3 — Parallel Expert Team

All agents work on their part simultaneously (async). No sequential dependency — maximum speed.

In [ ]:
market_researcher2 = Agent(
    name="MarketResearch", role="Market Research Analyst",
    goal="Identify target audience and their key pain points.",
    backstory="Consumer behaviour specialist with 12 years in B2B marketing.", verbose=False
)
copywriter = Agent(
    name="Copywriter", role="Senior Copywriter",
    goal="Write compelling marketing copy that converts.",
    backstory="Award-winning copywriter known for punchy, action-driven messaging.", verbose=False
)
social_media = Agent(
    name="SocialMedia", role="Social Media Strategist",
    goal="Create a social media strategy with platform-specific content.",
    backstory="Growth hacker who has scaled multiple B2B SaaS brands on LinkedIn and X.", verbose=False
)
seo_specialist = Agent(
    name="SEOSpecialist", role="SEO Content Strategist",
    goal="Identify keywords and SEO strategy to maximise organic reach.",
    backstory="Technical SEO expert with a track record in competitive technology niches.", verbose=False
)

async def parallel_campaign(product: str) -> dict:
    'Run all marketing agents in parallel, then synthesise.'
    print(f"\n🚀 Launching parallel marketing team for: '{product}'")
    t0 = time.perf_counter()

    audience, copy, social, seo = await asyncio.gather(
        market_researcher2.arun(f"Who is the target audience for: {product}? List 3 personas."),
        copywriter.arun(f"Write a tagline and 3 key messages for: {product}"),
        social_media.arun(f"Create a 1-week social media plan for launching: {product}"),
        seo_specialist.arun(f"List 10 target keywords and an SEO strategy for: {product}")
    )

    parallel_time = time.perf_counter() - t0
    print(f"✅ All 4 agents completed in {parallel_time:.2f}s (vs ~{parallel_time*4:.1f}s sequential)")

    campaign_brief = call_llm(
        f"""Compile a unified marketing campaign brief for: '{product}'

Audience Analysis:\n{audience}
Copy & Messaging:\n{copy}
Social Media Plan:\n{social}
SEO Strategy:\n{seo}

Format as a structured campaign brief with sections.""",
        system="You are a Chief Marketing Officer compiling a campaign brief."
    )

    return {"audience": audience, "copy": copy, "social": social,
            "seo": seo, "brief": campaign_brief}

campaign = await parallel_campaign("FlowAI — an AI-powered project management platform for remote teams")
print(f"\n{'='*60}\nCAMPAIGN BRIEF\n{'='*60}")
print(campaign["brief"])

## 6. Collaboration Model 4 — Debate & Consensus

Agents with different perspectives argue their position. A judge agent evaluates arguments and reaches a final decision.

In [ ]:
def debate_and_consensus(question: str, n_rounds: int = 2) -> dict:
    'Multi-agent debate:'
    print(f"\n🗣️  DEBATE: {question}")
    print(f"{'='*60}")

    debaters = [
        {"name": "Optimist",   "stance": "strongly in favour",
         "persona": "technology optimist who believes in transformative potential"},
        {"name": "Sceptic",    "stance": "cautiously against",
         "persona": "risk-focused analyst who prioritises evidence and caution"},
        {"name": "Pragmatist", "stance": "balanced and practical",
         "persona": "pragmatic strategist focused on implementation realities"},
    ]

    print("\n[ROUND 1] Opening positions:")
    positions = {}
    for d in debaters:
        position = call_llm(
            f"State your position on this question in 3-4 sentences: {question}",
            system=f"You are {d['name']}, a {d['persona']}. You are {d['stance']} on this topic. Be persuasive."
        )
        positions[d["name"]] = position
        print(f"\n  [{d['name']}]: {position[:200]}...")

    for round_num in range(2, n_rounds + 2):
        print(f"\n[ROUND {round_num}] Rebuttals:")
        all_positions = "\n\n".join(f"{k}: {v}" for k, v in positions.items())

        for d in debaters:
            others = "\n".join(f"{k}: {v}" for k, v in positions.items() if k != d["name"])
            rebuttal = call_llm(
                f"Other agents' positions:\n{others}\n\nRespond with a rebuttal in 2-3 sentences.",
                system=f"You are {d['name']}, a {d['persona']}. Be persuasive and concise."
            )
            positions[d["name"]] = rebuttal
            print(f"\n  [{d['name']} rebuttal]: {rebuttal[:150]}...")

    print("\n[JUDGE] Evaluating all arguments...")
    all_args = "\n\n".join(f"{k}:\n{v}" for k, v in positions.items())
    verdict = call_llm(
        f"""Question: {question}

Evaluate these arguments and provide:
1. Which argument was strongest and why
2. A balanced consensus position (3-4 sentences)
3. A final recommendation

Arguments:\n{all_args}""",
        system="You are an impartial expert judge. Be fair, evidence-based, and decisive."
    )
    print(f"\n[VERDICT]:\n{verdict}")

    return {"question": question, "positions": positions, "verdict": verdict}

debate_result = debate_and_consensus(
    "Should enterprises adopt AI agents for autonomous decision-making in financial operations?",
    n_rounds=1
)
print(f"\n{'='*60}\nFINAL VERDICT\n{'='*60}")
print(debate_result["verdict"])

## 7. Collaboration Model 5 — Critic-Reviewer Pattern

Creator builds → Reviewer evaluates → Creator revises Adds compliance, security, and quality checks between agents.

In [ ]:
def critic_reviewer_workflow(task: str, domain: str = "code") -> dict:
    'Creator → Compliance Reviewer → Security Reviewer → Final Reviser'
    print(f"\n{'='*60}")
    print(f"CRITIC-REVIEWER WORKFLOW: {domain.upper()}")
    print(f"{'='*60}")

    creator = Agent(
        name="Creator", role="Senior Software Engineer",
        goal="Write clean, functional code that meets requirements.",
        backstory="10 years of production Python experience at FAANG companies."
    )
    compliance_reviewer = Agent(
        name="ComplianceReviewer", role="Code Compliance Officer",
        goal="Ensure code follows PEP 8, has proper docstrings, type hints, and error handling.",
        backstory="Ex-Google code reviewer obsessed with standards and maintainability."
    )
    security_reviewer = Agent(
        name="SecurityReviewer", role="Application Security Engineer",
        goal="Identify security vulnerabilities: injection, input validation, data exposure.",
        backstory="OWASP expert and penetration tester with 8 years in application security."
    )
    reviser = Agent(
        name="Reviser", role="Lead Software Engineer",
        goal="Address all review feedback and produce the final, production-ready output.",
        backstory="Tech lead responsible for final code quality before production deployment."
    )

    print("\n[Stage 1] Creator generating initial output...")
    initial = creator.run(task)

    print("\n[Stage 2] Running compliance and security reviews...")
    compliance_feedback = compliance_reviewer.run(
        "Review the code below for compliance issues. List specific violations.",
        context=initial
    )
    security_feedback = security_reviewer.run(
        "Review the code below for security vulnerabilities. List specific issues.",
        context=initial
    )

    print(f"\n  Compliance issues found: {len(compliance_feedback.split(chr(10)))} lines of feedback")
    print(f"  Security issues found:   {len(security_feedback.split(chr(10)))} lines of feedback")

    print("\n[Stage 3] Reviser addressing all feedback...")
    final = reviser.run(
        "Apply ALL the feedback below to produce a final, production-ready version.",
        context=(
            f"Original code:\n{initial}\n\n"
            f"Compliance feedback:\n{compliance_feedback}\n\n"
            f"Security feedback:\n{security_feedback}"
        )
    )

    return {
        "initial": initial,
        "compliance_feedback": compliance_feedback,
        "security_feedback": security_feedback,
        "final": final
    }

cr_result = critic_reviewer_workflow(
    task="Write a Python function that accepts a username and password, validates them against a dict of users, and returns a JWT token on success.",
    domain="code"
)

print(f"\n{'='*60}\nFINAL PRODUCTION-READY CODE\n{'='*60}")
print(cr_result["final"])

## 8. Collaboration Model 6 — Hierarchical Multi-Level

Dept Manager → Team Leads → Specialists The department manager only talks to leads; leads talk to their specialists.

In [ ]:
class TeamLead:
    'Manages a sub-team of specialist agents.'
    def __init__(self, name: str, domain: str, specialists: List[Agent]):
        self.name        = name
        self.domain      = domain
        self.specialists = specialists

    def run(self, objective: str) -> str:
        print(f"\n  [TeamLead: {self.name}] Managing {len(self.specialists)} specialists")

        specialist_outputs = []
        for spec in self.specialists:
            output = spec.run(f"{self.domain} perspective on: {objective}")
            specialist_outputs.append(f"[{spec.name}]: {output}")

        combined = "\n\n".join(specialist_outputs)
        synthesis = call_llm(
            f"As {self.domain} team lead, synthesise your team's findings into a concise brief:\n\n{combined}",
            system=f"You are {self.name}, a {self.domain} team lead. Be precise and strategic."
        )
        print(f"  [TeamLead: {self.name}] ✅ Brief ready")
        return synthesis

class DepartmentHead:
    'Top-level orchestrator that coordinates team leads.'
    def __init__(self, name: str, teams: List[TeamLead]):
        self.name  = name
        self.teams = teams

    def run(self, goal: str) -> str:
        print(f"\n{'='*60}")
        print(f"HIERARCHICAL SYSTEM — Dept Head: {self.name}")
        print(f"Goal: {goal}")
        print(f"{'='*60}")

        team_briefs = []
        for team in self.teams:
            brief = team.run(goal)
            team_briefs.append(f"[{team.name} team]:\n{brief}")

        all_briefs = "\n\n".join(team_briefs)
        executive_summary = call_llm(
            f"Goal: {goal}\n\nTeam briefs:\n{all_briefs}\n\nWrite an executive summary and final recommendation.",
            system=f"You are {self.name}, a department head. Write for C-suite audience."
        )
        return executive_summary

tech_team = TeamLead(
    name="TechLead", domain="Technology",
    specialists=[
        Agent("ArchAgent", "Solutions Architect", "Assess technical feasibility", "Cloud architecture expert.", verbose=False),
        Agent("SecAgent",  "Security Engineer",   "Identify security concerns",   "AppSec specialist.", verbose=False),
    ]
)

business_team = TeamLead(
    name="BizLead", domain="Business",
    specialists=[
        Agent("FinAgent", "Financial Analyst", "Quantify costs and ROI",        "Enterprise finance expert.", verbose=False),
        Agent("MktAgent", "Market Analyst",    "Assess market opportunity",      "B2B SaaS market researcher.", verbose=False),
    ]
)

dept_head = DepartmentHead(
    name="ChiefProductOfficer",
    teams=[tech_team, business_team]
)

exec_summary = dept_head.run("Should we build an internal AI-powered data analytics platform or buy an existing SaaS solution?")
print(f"\n{'='*60}\nEXECUTIVE SUMMARY\n{'='*60}")
print(exec_summary)

## 9. Agent as a Tool — One Agent Calls Another

A parent agent invokes a specialist sub-agent as if calling a function. The sub-agent's entire workflow is hidden inside the tool interface.

In [ ]:
class AgentTool:
    'Wraps a specialist Agent as a callable tool.'
    def __init__(self, agent: Agent, tool_name: str, description: str):
        self.agent       = agent
        self.tool_name   = tool_name
        self.description = description

    def __call__(self, task: str, context: str = "") -> str:
        print(f"\n  🔧 [AgentTool: {self.tool_name}] → {self.agent.name}")
        return self.agent.run(task, context)

    def schema(self) -> dict:
        return {"name": self.tool_name, "description": self.description,
                "input": "task: str, context: str (optional)"}

legal_agent = Agent(
    name="LegalSpecialist", role="Corporate Lawyer",
    goal="Identify legal risks and compliance requirements.",
    backstory="Partner at a top-tier law firm specialising in technology contracts."
)
financial_agent = Agent(
    name="FinancialSpecialist", role="CFA-certified Financial Analyst",
    goal="Model financial projections and assess ROI.",
    backstory="20 years in corporate finance and M&A advisory."
)
technical_agent = Agent(
    name="TechnicalSpecialist", role="Principal Software Architect",
    goal="Assess technical complexity and provide implementation estimates.",
    backstory="Built 5 enterprise platforms from scratch at Fortune 500 companies."
)

legal_tool     = AgentTool(legal_agent,     "consult_legal",     "Get legal analysis and compliance review from a corporate lawyer.")
financial_tool = AgentTool(financial_agent, "consult_financial", "Get financial modelling and ROI analysis from a CFA analyst.")
technical_tool = AgentTool(technical_agent, "consult_technical", "Get technical feasibility and complexity assessment from an architect.")

class OrchestratorAgent:
    'Master agent that uses other agents as tools.'
    def __init__(self, tools: List[AgentTool]):
        self.tools = {t.tool_name: t for t in tools}

    def decide_tools(self, query: str) -> list:
        tool_descriptions = "\n".join(
            f"- {t.tool_name}: {t.description}" for t in self.tools.values()
        )
        raw = call_llm(
            f"""Given this query, which tools should be invoked? Return a JSON array of tool names.

Query: {query}
Available tools:\n{tool_descriptions}

Return ONLY a JSON array of tool names, e.g. ["consult_legal", "consult_financial"]""",
            system="You are a routing orchestrator. Return only valid JSON."
        )
        return json.loads(re.sub(r"```[\w]*", "", raw).strip())

    def run(self, query: str) -> str:
        print(f"\n{'='*60}\nORCHESTRATOR: {query}\n{'='*60}")

        selected = self.decide_tools(query)
        print(f"\n📋 Selected tools: {selected}")

        tool_results = {}
        for tool_name in selected:
            if tool_name in self.tools:
                result = self.tools[tool_name](query)
                tool_results[tool_name] = result

        results_text = "\n\n".join(f"[{k}]:\n{v}" for k, v in tool_results.items())
        final = call_llm(
            f"Query: {query}\n\nSpecialist inputs:\n{results_text}\n\nSynthesize a final executive recommendation.",
            system="You are a strategic advisor synthesising expert opinions."
        )
        return final

orchestrator = OrchestratorAgent(tools=[legal_tool, financial_tool, technical_tool])
recommendation = orchestrator.run(
    "We are considering acquiring a small AI startup for $5M. What do we need to assess?"
)
print(f"\n{'='*60}\nFINAL RECOMMENDATION\n{'='*60}")
print(recommendation)

## 10. Inter-Agent Communication Protocol

Production systems need structured messages between agents. We implement a typed message protocol with routing and metadata.

In [ ]:
import uuid
from datetime import datetime

@dataclass
class AgentMessage:
    'Typed, traceable message for inter-agent communication.'
    sender:      str
    recipient:   str
    content:     str
    msg_type:    str = "task"
    priority:    int = 1
    metadata:    Dict = field(default_factory=dict)
    msg_id:      str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    timestamp:   str = field(default_factory=lambda: datetime.now().isoformat())
    parent_id:   Optional[str] = None

    def __str__(self):
        return (
            f"[{self.timestamp[:19]}] {self.msg_id} "
            f"{self.sender} → {self.recipient} "
            f"[{self.msg_type.upper()}|P{self.priority}]: "
            f"{self.content[:80]}{'...' if len(self.content) > 80 else ''}"
        )

class MessageBus:
    'Central message bus for agent communication.'
    def __init__(self):
        self.log: List[AgentMessage] = []

    def send(self, msg: AgentMessage):
        self.log.append(msg)
        print(f"  📨 {msg}")

    def get_thread(self, msg_id: str) -> List[AgentMessage]:
        'Get all messages in a conversation thread.'
        return [m for m in self.log if m.msg_id == msg_id or m.parent_id == msg_id]

    def summary(self):
        print(f"\n📊 Message Bus: {len(self.log)} total messages")
        types = {}
        for m in self.log:
            types[m.msg_type] = types.get(m.msg_type, 0) + 1
        for t, c in types.items():
            print(f"   {t}: {c}")

class CommunicatingAgent:
    'Agent that communicates via structured messages.'
    def __init__(self, name: str, role: str, bus: MessageBus):
        self.name = name
        self.role = role
        self.bus  = bus

    def send(self, recipient: str, content: str, msg_type: str = "task",
             priority: int = 1, parent_id: str = None) -> AgentMessage:
        msg = AgentMessage(
            sender=self.name, recipient=recipient, content=content,
            msg_type=msg_type, priority=priority, parent_id=parent_id
        )
        self.bus.send(msg)
        return msg

    def process(self, task: str) -> str:
        return call_llm(task, system=f"You are {self.name}, a {self.role}. Be precise.")

bus = MessageBus()

pm      = CommunicatingAgent("ProjectManager", "project manager", bus)
dev     = CommunicatingAgent("Developer",      "senior developer", bus)
qa      = CommunicatingAgent("QATester",       "QA engineer", bus)

print("=== STRUCTURED INTER-AGENT COMMUNICATION ===")

task_msg = pm.send("Developer", "Write a Python function to validate email addresses.", msg_type="task", priority=2)

code = dev.process("Write a Python function to validate email addresses. Include type hints.")
result_msg = dev.send("ProjectManager", f"Code complete:\n{code[:200]}...",
                      msg_type="result", parent_id=task_msg.msg_id)

qa_msg = pm.send("QATester", f"Please test this code: {code[:100]}...",
                 msg_type="handoff", parent_id=task_msg.msg_id)

qa_result = qa.process(f"Review this for test coverage and edge cases: {code[:300]}")
qa.send("ProjectManager", f"QA complete: {qa_result[:150]}...",
        msg_type="result", parent_id=qa_msg.msg_id)

bus.summary()

## 11. Full End-to-End — Software Development Team

A complete multi-agent software development pipeline: Requirements → Design → Code → Test → Documentation

In [ ]:
async def software_dev_team(feature_request: str) -> dict:
    'Full software development multi-agent workflow.'

    print(f"\n{'='*60}")
    print(f"SOFTWARE DEV TEAM: {feature_request}")
    print(f"{'='*60}")

    ba = Agent("BusinessAnalyst", "Business Analyst",
               "Translate feature requests into clear technical requirements.",
               "Expert at writing user stories and acceptance criteria.", verbose=False)
    architect = Agent("SysArchitect", "System Architect",
                      "Design the technical solution and data model.",
                      "10 years designing scalable Python backend systems.", verbose=False)
    developer = Agent("Developer", "Senior Python Developer",
                      "Write clean, type-hinted, documented Python code.",
                      "Specialist in FastAPI, SQLAlchemy, and Pydantic.", verbose=False)
    tester = Agent("QATester", "QA Engineer",
                   "Write comprehensive pytest test cases.",
                   "TDD advocate who writes tests before reviewing code.", verbose=False)
    doc_writer = Agent("DocWriter", "Technical Writer",
                       "Write clear API documentation and usage examples.",
                       "Specialist in Swagger/OpenAPI and developer documentation.", verbose=False)

    print("\n[Stage 1] Business Analyst writing requirements...")
    requirements = ba.run(
        f"Write user stories and acceptance criteria for: {feature_request}"
    )

    print("[Stage 2] Architect designing solution...")
    design = architect.run(
        "Create a technical design: data models, API endpoints, and key decisions.",
        context=requirements
    )

    print("[Stage 3] Developer and Tester working in parallel...")
    t0 = time.perf_counter()
    code, tests = await asyncio.gather(
        developer.arun(
            "Write the complete Python implementation with type hints and docstrings.",
            context=f"Requirements:\n{requirements}\n\nDesign:\n{design}"
        ),
        tester.arun(
            "Write pytest test cases covering happy path, edge cases, and error conditions.",
            context=f"Requirements:\n{requirements}\n\nDesign:\n{design}"
        )
    )
    print(f"   ⚡ Parallel stage completed in {time.perf_counter()-t0:.2f}s")

    print("[Stage 4] Doc Writer creating documentation...")
    docs = doc_writer.run(
        "Write API documentation with usage examples.",
        context=f"Code:\n{code[:500]}...\n\nTests:\n{tests[:300]}..."
    )

    print("\n✅ All stages complete!")
    return {
        "requirements": requirements,
        "design":       design,
        "code":         code,
        "tests":        tests,
        "docs":         docs
    }

dev_result = await software_dev_team(
    "Build a user authentication API endpoint with JWT tokens, rate limiting, and audit logging"
)

print(f"\n{'='*60}\nDELIVERABLES SUMMARY\n{'='*60}")
for key, value in dev_result.items():
    print(f"\n── {key.upper()} ──")
    print(value[:300] + "..." if len(value) > 300 else value)

## 12. Collaboration Model Comparison

In [ ]:
comparison = """
╔══════════════════════╦══════════════╦════════════════╦═══════════════════════════╗
║ Model                ║ Speed        ║ Complexity     ║ Best For                  ║
╠══════════════════════╬══════════════╬════════════════╬═══════════════════════════╣
║ Sequential Handoff   ║ Medium       ║ Low            ║ Ordered pipelines where   ║
║                      ║              ║                ║ each step builds on prev  ║
╠══════════════════════╬══════════════╬════════════════╬═══════════════════════════╣
║ Supervisor-Worker    ║ Medium       ║ Medium         ║ Delegating diverse tasks  ║
║                      ║              ║                ║ to specialist teams       ║
╠══════════════════════╬══════════════╬════════════════╬═══════════════════════════╣
║ Parallel Expert Team ║ Fast         ║ Medium         ║ Independent domains that  ║
║                      ║              ║                ║ need parallel execution   ║
╠══════════════════════╬══════════════╬════════════════╬═══════════════════════════╣
║ Debate & Consensus   ║ Slow         ║ High           ║ Strategic decisions where ║
║                      ║              ║                ║ multiple views matter     ║
╠══════════════════════╬══════════════╬════════════════╬═══════════════════════════╣
║ Critic-Reviewer      ║ Medium       ║ Medium         ║ Code, content, plans that ║
║                      ║              ║                ║ need compliance/security  ║
╠══════════════════════╬══════════════╬════════════════╬═══════════════════════════╣
║ Hierarchical         ║ Slow         ║ High           ║ Enterprise-scale systems  ║
║                      ║              ║                ║ with structured authority ║
╠══════════════════════╬══════════════╬════════════════╬═══════════════════════════╣
║ Agent as Tool        ║ Variable     ║ Medium         ║ Specialist capabilities   ║
║                      ║              ║                ║ hidden behind clean APIs  ║
╚══════════════════════╩══════════════╩════════════════╩═══════════════════════════╝

⚠️  KEY TRADE-OFF: More agents = more specialisation + more LLM calls + more latency.
   Start simple (sequential) and add complexity only when quality demands it.
"""
print(comparison)

## 13. Exercises

- **Exercise 1 — Supply Chain:** Build a parallel multi-agent system where three agents simultaneously simulate: (a) supplier inventory, (b) shipping costs, (c) demand forecast — then a 4th agent optimises order quantities.

- **Exercise 2 — Three-Round Debate:** Extend `debate_and_consensus` to 3 rounds. Add a 4th debater: `"Devil's Advocate"` whose role is always to challenge whichever position is currently winning.

- **Exercise 3 — Agent Registry:** Build a dynamic `AgentRegistry` class that allows agents to register themselves at runtime. The `SupervisorAgent` should query the registry to find the best agent for each sub-task, rather than having a fixed list.

- **Exercise 4 — Failure Escalation:** Add fault tolerance to `sequential_pipeline`: if any agent's output is less than 50 words or contains "I cannot", automatically escalate to a backup agent with a more specific prompt.

- **Exercise 5 — Full Stack Team:** Extend `software_dev_team` with two additional agents: a `SecurityReviewer` (runs in parallel with QA) and a `DevOps` agent (writes a Dockerfile and CI/CD config after code is complete).

**Next:** Chapter 8 — Memory Systems for Agents